# Heat Demand 2024 (Plymouth, LSOA)

This notebook converts LSOA gas consumption into useful residential heating demand and saves the final demand layer.


## 1. Imports



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

## 2. configure paths and assumptionn


In [ ]:
PROJECT_DIR = next(candidate
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (candidate / "01_Data").exists()
                   )

if PROJECT_DIR.name == "02_Code":
    PROJECT_DIR = PROJECT_DIR.parent

In [ ]:
PROCESSED_DATA_DIR = PROJECT_DIR / "01_Data" / "Processed"

PROCESSED_ENERGY_DIR = PROCESSED_DATA_DIR / "Energy"
PROCESSED_EPC_DIR = PROCESSED_DATA_DIR / "EPC"
PROCESSED_ENERGY_DIR.mkdir(parents=True, exist_ok=True)

ENERGY_GPKG = PROCESSED_ENERGY_DIR / "plymouth_energy_consumption_2024.gpkg"
ENERGY_LAYER = "lsoa_energy_consumption_2024"

EPC_LSOA_CSV = PROCESSED_EPC_DIR / "plymouth_offgas_energy_epc_lsoa_2024.csv"

HEAT_DEMAND_CSV = PROCESSED_ENERGY_DIR / "plymouth_lsoa_heat_demand_2024.csv"
HEAT_DEMAND_GPKG = PROCESSED_ENERGY_DIR / "plymouth_heat_demand_2024.gpkg"
HEAT_DEMAND_LAYER = "lsoa_heat_demand_2024"


SENSITIVITY_CSV = PROCESSED_ENERGY_DIR / "plymouth_heat_demand_sensitivity_2024.csv"

# Assumptions
HEAT_SHARE_OF_GAS = 0.975
BOILER_EFFICIENCY = 0.84

print("Project directory:", PROJECT_DIR)


## 3. Load processed energy and EPC data


In [ ]:
energy_gdf = gpd.read_file(ENERGY_GPKG, layer=ENERGY_LAYER)

required_cols = ["LSOA_code","LSOA","gas_consumption_kwh","domestic_properties",
                 "domestic_gas_meters_offgas_file","off_gas_properties",
                 "off_gas_property_share","area_km2",]
missing_cols = [col for col in required_cols if col not in energy_gdf.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in energy layer: {missing_cols}")

print("Energy layer shape:", energy_gdf.shape)
print("CRS:", energy_gdf.crs)
print("Unique LSOAs:", energy_gdf["LSOA_code"].nunique())
energy_gdf[required_cols].head()


In [ ]:
# Load EPC LSOA summary data
epc_lsoa = pd.read_csv(EPC_LSOA_CSV)


In [ ]:
# Inspect EPC LSOA columns
epc_lsoa.columns.tolist()


In [ ]:
# Inspect EPC LSOA row and column count
epc_lsoa.shape


## 4. Estimate gas useful heat demand


In [ ]:
heat_gdf = energy_gdf.copy()

# Electricity is not used for this heat-demand estimate, so remove those columns from this output.
electricity_cols = [col for col in heat_gdf.columns if col.startswith("electricity_")]
heat_gdf = heat_gdf.drop(columns=electricity_cols)

#calculate fraction of gas used for residential heating
heat_gdf["gas_heating_fuel_input_kwh"] = (heat_gdf["gas_consumption_kwh"] * HEAT_SHARE_OF_GAS)

#calculate useful heat
heat_gdf["gas_useful_heat_kwh"] = (heat_gdf["gas_heating_fuel_input_kwh"] * BOILER_EFFICIENCY)

#calculate useful heat per km2
heat_gdf["gas_useful_heat_mwh_per_km2"] = ((heat_gdf["gas_useful_heat_kwh"] / 1_000) / heat_gdf["area_km2"])

## 5. Add EPC-based off-gas heat demand


In [ ]:
heat_gdf = heat_gdf.merge(epc_lsoa[["lsoa21cd","OFFGAS_EPC_COUNT","ADJUSTED_MEDIAN_HEAT_PER_PROPERTY_KWH","HEAT_ESTIMATE_SOURCE"]],
                          left_on="LSOA_code", right_on="lsoa21cd",how="left"
                          )

heat_gdf = heat_gdf.drop(columns="lsoa21cd")

#calculate useful heat demand from off-gas properties
heat_gdf["offgas_useful_heat_kwh"] = (heat_gdf["off_gas_properties"]* heat_gdf["ADJUSTED_MEDIAN_HEAT_PER_PROPERTY_KWH"])

#combine gas and off-gas useful heat
heat_gdf["total_useful_heat_kwh"] = heat_gdf["gas_useful_heat_kwh"]+ heat_gdf["offgas_useful_heat_kwh"]

heat_gdf["total_useful_heat_mwh"] = heat_gdf["total_useful_heat_kwh"] / 1_000

heat_gdf["total_useful_heat_gwh"] = heat_gdf["total_useful_heat_kwh"] / 1_000_000

heat_gdf["total_useful_heat_mwh_per_km2"] = heat_gdf["total_useful_heat_mwh"]/ heat_gdf["area_km2"]

## 6. Summarise citywide heat demand


In [ ]:
print("Domestic gas input GWh:", heat_gdf["gas_consumption_kwh"].sum() / 1_000_000)
print("Gas heating fuel input GWh:", heat_gdf["gas_heating_fuel_input_kwh"].sum() / 1_000_000)
print("Gas useful heat GWh:", heat_gdf["gas_useful_heat_kwh"].sum() / 1_000_000)
print("Estimated off-gas useful heat GWh:",heat_gdf["offgas_useful_heat_kwh"].sum() / 1_000_000)
print("Estimated total useful heat GWh:", heat_gdf["total_useful_heat_gwh"].sum())

print("Citywide off-gas property share:",heat_gdf["off_gas_properties"].sum() / heat_gdf["domestic_properties"].sum())


## 7. Prepare QGIS heat-demand layer


In [ ]:
heat_gdf["off_gas_property_percent"] = heat_gdf["off_gas_property_share"] * 100

# Columns needed for the final QGIS heat-demand layer
qgis_columns = [
    "LSOA_code",
    "LSOA",
    "total_useful_heat_gwh",
    "total_useful_heat_mwh",
    "total_useful_heat_mwh_per_km2",
    "off_gas_property_percent",
    "offgas_useful_heat_kwh",
    "geometry"
]

qgis_heat_gdf = heat_gdf[qgis_columns].copy()

## 8. Save processed heat-demand outputs


In [ ]:
# Keep geometry in the GeoPackage and write a flat table without geometry for GIS joins.
non_geometry_cols = [col for col in heat_gdf.columns if col != heat_gdf.geometry.name]

heat_gdf[non_geometry_cols].to_csv(HEAT_DEMAND_CSV, index=False)

if HEAT_DEMAND_GPKG.exists():
    HEAT_DEMAND_GPKG.unlink()

heat_gdf.to_file(
    HEAT_DEMAND_GPKG,
    layer=HEAT_DEMAND_LAYER,
    driver="GPKG",
)

